In [ ]:
!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage
#!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

train_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train"
test_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val"

BATCH_SIZE = 64
IMG_SIZE = (224,224)

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)


test_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)


print("Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:\n")
print("class_names = [")
for name in train_ds.class_names:
    print(f'    "{name}",')
print("]")

In [ ]:
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.05),
    layers.RandomContrast(0.05),
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(38, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

model.summary()

In [ ]:
print("--starting first stage--")

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

In [ ]:
# print("--preparing model for second stage--")

# unfreezeModel = model.layers[4]
# unfreezeModel.trainable = True

# for layer in unfreezeModel.layers[:-30]:
#     layer.trainable = False

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

In [ ]:
# print("--starting second stage - fine-tuning--")

# history_f = model.fit(
#     train_ds,
#     validation_data = test_ds,
#     epochs=7
# )

In [ ]:
# Zapisanie wyszkolonego modelu 
model_save_path = '/kaggle/working/plant_disease_detector_v3.keras'
model.save(model_save_path)

print(f"Model został pomyślnie zapisany w: {model_save_path}")